In [11]:
import pandas as pd
import numpy as np
import time
import math
import os
import xgboost as xgb

import lightgbm as lgb
from lightgbm import LGBMRanker

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from tqdm import tqdm
print("fim")
tqdm.pandas()

fim


In [2]:
def load_data (split):
    df_train = pd.read_csv(f"./splits/{split}/train_pares.csv")
    df_val = pd.read_csv(f"./splits/{split}/val_pares.csv")
    df_test = pd.read_csv(f"./splits/{split}/test_pares.csv")

    return df_train, df_val, df_test

In [3]:
def ranking_metrics(group):
    group_id = group.name

    group = group.sort_values("score", ascending=False)

    labels = group["label"].values
    relevant_positions = np.where(labels == 1)[0]

    n_relevant = len(relevant_positions)

    #se não há relevantes, estoura erro
    if n_relevant == 0:
        #print(f"Nenhum parente encontrado id={group_id}")
        return pd.Series({
            "id": group_id,
            "recall@1": np.nan,
            "recall@5": np.nan,
            "recall@10": np.nan,
            "recall@15": np.nan,
            "recall@20": np.nan,
            "mrr": np.nan,
            "count": np.nan
        })

    # Recall@1
    recall_at_1 = int(labels[0] == 1) / n_relevant

    # Recall@5
    recall_at_5 = labels[:5].sum() / n_relevant

    # Recall@10
    recall_at_10 = labels[:10].sum() / n_relevant

    # Recall@10
    recall_at_15 = labels[:15].sum() / n_relevant

    # Recall@10
    recall_at_20 = labels[:20].sum() / n_relevant

    # MRR
    first_relevant_rank = relevant_positions[0] + 1
    mrr = 1.0 / first_relevant_rank

    return pd.Series({
        "id": group_id,
        "recall@1": recall_at_1,
        "recall@5": recall_at_5,
        "recall@10": recall_at_10,
        "recall@15": recall_at_15,
        "recall@20": recall_at_20,
        "mrr": mrr,
        "count": len(labels)
    })

In [12]:
#nova célula 4 com função inferencia 
def inferencia(label, df_test, split_path, features, model, mode="ranker"):
    """Realiza inferência e calcula métricas de ranking"""
    
    X_test = df_test[features].copy()
    
    if mode == "heuristic":
        cols_to_fill = ["age_score", "addr_score", "occ_score"]
        X_test[cols_to_fill] = X_test[cols_to_fill].fillna(0)
        scores = (
            0.50 * X_test["age_score"] +
            0.40 * X_test["addr_score"] +
            0.10 * X_test["occ_score"]
        )
    
    elif mode == "classifier":
        cols_to_fill = ["age_score", "addr_score", "occ_score", "addr_exato"]
        X_test[cols_to_fill] = X_test[cols_to_fill].fillna(0)
        scores = model.predict_proba(X_test)[:, 1]
    
    elif mode == "ranker":
        scores = model.predict(X_test)
    
    df_test_copy = df_test.copy()
    df_test_copy["score"] = scores
    
    df_results = df_test_copy.groupby(["ID"]).apply(ranking_metrics)
    df_results = df_results.reset_index(drop=True)
    df_results.to_csv(f"{split_path}/{label}.csv", index=False)
    
    return df_results





In [13]:
#one hot encoding 

def create_one_hot_encoding(df):
    relation_categories = [
        'Spouse', 'Biological child', 'Adopted child', 'Child-in-law',
        'Parent-in-law', 'Parent', 'Grandchild', 'Grandparent', 'Sibling',
        'Nephew/niece', 'Stepchild', 'Stepparent', 'Sibling-in-law'
    ]
    
    #coloca 0 ou 1 nas categorias 
    for category in relation_categories:
        col_name = f"is_{category.replace(' ', '').replace('/', '_')}"
        df[col_name] = (df['RELATED'] == category).astype(int)
    
    return df

def prepare_features(df_train, df_valid, df_test):
    #prepara as features para os três datasets
    df_train = create_one_hot_encoding(df_train)
    df_valid = create_one_hot_encoding(df_valid)
    df_test = create_one_hot_encoding(df_test)
    
    return df_train, df_valid, df_test


In [14]:
#célula nova do xgboost
def train_xgb_ranker(df_train, df_valid, features):
    """Treina modelo XGBoost Ranker"""
    
    df_train = df_train.sort_values("ID")
    df_valid = df_valid.sort_values("ID")
    
    X_train = df_train[features]
    Y_train = df_train["label"]

    X_valid = df_valid[features]
    Y_valid = df_valid["label"]
   
    group_train = (
        df_train
        .groupby("ID")
        .size()
        .values
    )
    
    group_valid = (
        df_valid
        .groupby("ID")
        .size()
        .values
    )

    model = xgb.XGBRanker(
        objective="rank:ndcg",
        eval_metric="ndcg@10",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        Y_train,
        group=group_train,
        eval_set=[(X_valid, Y_valid)],
        eval_group=[group_valid],
        verbose=20
    )

    return model

In [15]:
#célula nova do lgbm

def train_lgbm_ranker(df_train, df_valid, features):
    """Treina modelo LightGBM Ranker"""
    
    df_train = df_train.sort_values("ID")
    df_valid = df_valid.sort_values("ID")
    
    X_train = df_train[features]
    Y_train = df_train["label"]

    X_valid = df_valid[features]
    Y_valid = df_valid["label"]
    
    #grupos para ranking (precisa ser lista para LightGBM)
    group_train = df_train.groupby("ID").size().tolist()
    group_valid = df_valid.groupby("ID").size().tolist()

    model = LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_train,
        Y_train,
        group=group_train,
        eval_set=[(X_valid, Y_valid)],
        eval_group=[group_valid],
        eval_at=[1, 5, 10],
        callbacks=[
            lgb.log_evaluation(period=20)
        ]
    )

    return model

In [16]:
#célula nova do reg_log 

def train_reg_log_classifier(df_train, df_valid, features):
    """Treina modelo de Regressão Logística"""
    
    #colunas sem one code 
    cols_to_fill = [
        "age_score",
        "addr_score",
        "occ_score",
        "addr_exato"
    ]
    
    # dados de treino
    X_train = df_train[features].copy()
    X_train[cols_to_fill] = X_train[cols_to_fill].fillna(0)
    Y_train = df_train["label"]
    
    # dados de validação
    X_valid = df_valid[features].copy()
    X_valid[cols_to_fill] = X_valid[cols_to_fill].fillna(0)
    Y_valid = df_valid["label"]
    
    #calcular peso das classes (balanceamento)
    n_pos = (Y_train == 1).sum()
    n_neg = (Y_train == 0).sum()
    
    class_weight = {
        0: 1.0,
        1: n_neg / n_pos  # Dar mais peso aos positivos (são minoria)
    }
    

    model = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        class_weight=class_weight,
        n_jobs=-1
    )
    
    #pipeline com normalização
    scaler = StandardScaler()
    pipeline = Pipeline([
        ("scaler", scaler),
        ("clf", model)
    ])
    
    #treino
    pipeline.fit(X_train, Y_train)
    
    #validar
    y_valid_proba = pipeline.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(Y_valid, y_valid_proba)
    print(f"  Validation AUC: {auc:.4f}")
    
    return pipeline

In [17]:
#célula nova com heurística, xgboost e lgbm

OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)


for split in range(1, 6):
    print(f"\n{'='*50}")
    print(f"SPLIT {split}")
    print(f"{'='*50}\n")
    
    split_path = OUTPUT_PATH+f"/{split}"
    os.makedirs(split_path, exist_ok=True)
    
    # Carregar dados
    print(f"Loading Data {split}")
    df_train, df_val, df_test = load_data(split)
    
    # Shuffle dos dados de treino
    print(f"Shuffling train data {split}")
    df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Criar one-hot encoding
    print(f"Creating one-hot encoding {split}")
    df_train = pd.get_dummies(df_train, columns=['RELATED'], prefix='is')
    df_val = pd.get_dummies(df_val, columns=['RELATED'], prefix='is')
    df_test = pd.get_dummies(df_test, columns=['RELATED'], prefix='is')
    
    # Garantir que todos os dataframes tenham as mesmas colunas
    all_columns = list(set(df_train.columns) | set(df_val.columns) | set(df_test.columns))
    for col in all_columns:
        if col not in df_train.columns:
            df_train[col] = 0
        if col not in df_val.columns:
            df_val[col] = 0
        if col not in df_test.columns:
            df_test[col] = 0
    
    # Atualizar features com as novas colunas is_*
    is_columns = [col for col in df_train.columns if col.startswith('is_')]
    features_rank = ['age_score', 'addr_score', 'addr_exato', 'occ_score'] + is_columns


    # ==================== XGBoost Ranker ====================
    label = "xgb_ranker"
    print(f"Training XGBoost Ranker {split}")
    xgb_model = train_xgb_ranker(df_train, df_val, features_rank)
    
    print(f"Inference XGBoost Ranker {split}")
    inferencia(label, df_test, split_path, features_rank, xgb_model, mode="ranker")

    # ==================== LightGBM Ranker ====================
    label = "lgbm_ranker"
    print("Training model LGBM", split)
    lgbm_model = train_lgbm_ranker(df_train, df_val, features_rank)
    print(f"Inference LightGBM Ranker {split}")
    inferencia(label, df_test, split_path, features_rank, lgbm_model, mode="ranker")

    # ==================== Regressão Logística  ====================
     
    features_reg_log = features_rank + [
        'missing_age',
        'missing_addr',
        'missing_occ',
    ]
    label = "reg_log"
    print(f"Training model Regressão Logistica {split}")
    reg_log_model = train_reg_log_classifier(df_train, df_val, features_reg_log)
    print(f"Inference model Regressão Logistica {split}")
    inferencia(label, df_test, split_path, features_reg_log, reg_log_model, mode="classifier")

    # ==================== Heuristica  ====================
    features_heuristic = [
        "age_score",
        "addr_score",
        "occ_score",
    ]
    label = "heuristica"
    print("Inference heuristica", split)
    inferencia(label, df_test, split_path, features_heuristic, None, mode="heuristic")


print("fim")


SPLIT 1

Loading Data 1
Shuffling train data 1
Creating one-hot encoding 1
Training XGBoost Ranker 1
[0]	validation_0-ndcg@10:0.58030
[20]	validation_0-ndcg@10:0.58030
[40]	validation_0-ndcg@10:0.58030
[60]	validation_0-ndcg@10:0.58030
[80]	validation_0-ndcg@10:0.58030
[100]	validation_0-ndcg@10:0.58030
[120]	validation_0-ndcg@10:0.58030
[140]	validation_0-ndcg@10:0.58030
[160]	validation_0-ndcg@10:0.58030
[180]	validation_0-ndcg@10:0.58030
[200]	validation_0-ndcg@10:0.58030
[220]	validation_0-ndcg@10:0.58030
[240]	validation_0-ndcg@10:0.58030
[260]	validation_0-ndcg@10:0.58030
[280]	validation_0-ndcg@10:0.58030
[299]	validation_0-ndcg@10:0.58030
Inference XGBoost Ranker 1
Training model LGBM 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000185 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_w

/tmp/ipykernel_15468/630035221.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_results = df_test_copy.groupby(["ID"]).apply(ranking_metrics)


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped t

/tmp/ipykernel_15468/630035221.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_results = df_test_copy.groupby(["ID"]).apply(ranking_metrics)
/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provid

In [12]:

LABELS = ["heuristica", "reg_log", "xgboost_ranker", "lgbm_ranker"]

SPLITS = [1,2,3,4,5]
OUTPUT_PATH = "output" 
metrics_to_print = ["recall@1", "recall@5", "recall@10", "recall@15", "recall@20", "mrr"]
result = []
for label in LABELS:

    for split in SPLITS:
        split_path = f"{OUTPUT_PATH}/{split}"
        file_path = f"{split_path}/{label}.csv"
        df_results = pd.read_csv(file_path)
        
        metrics = df_results[metrics_to_print].mean()
        metrics["split"] = split
        metrics["label"] = label
        result.append(metrics)

df = pd.DataFrame(result)



print(f"\n{'='*60}")
print("RESULTADOS CONSOLIDADOS")
print(f"{'='*60}\n")
print(df.to_string())

# Mostrar média por método
print(f"\n{'='*60}")
print("MÉDIA POR MÉTODO")
print(f"{'='*60}\n")
df_avg = df.groupby("label")[metrics_to_print].mean()
print(df_avg.to_string())




RESULTADOS CONSOLIDADOS

    recall@1  recall@5  recall@10  recall@15  recall@20       mrr  split        label
0   0.337450  0.610387   0.704496   0.754714   0.787550  0.464724    1.0   heuristica
1   0.337131  0.610738   0.705784   0.755250   0.788003  0.465060    2.0   heuristica
2   0.334815  0.608180   0.702218   0.752781   0.786766  0.462701    3.0   heuristica
3   0.335138  0.608393   0.701168   0.752087   0.785166  0.462715    4.0   heuristica
4   0.337141  0.609694   0.703458   0.754039   0.787290  0.464517    5.0   heuristica
5   0.798911  0.967986   0.986446   0.991733   0.994298  0.874234    1.0      reg_log
6   0.798547  0.967769   0.986082   0.991510   0.994017  0.873548    2.0      reg_log
7   0.797654  0.968860   0.986497   0.991670   0.994240  0.873518    3.0      reg_log
8   0.797930  0.967795   0.986050   0.991491   0.994119  0.873388    4.0      reg_log
9   0.799084  0.967036   0.985349   0.991102   0.993749  0.874024    5.0      reg_log
10  0.805028  0.970372   0.9